<a href="https://colab.research.google.com/github/thesagepip/people_analytics_ai_portfolio/blob/main/01_Strategic_Workforce_Intelligence_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# STRATEGIC WORKFORCE INTELLIGENCE PROJECT
# STEP 1: PROJECT SETUP
# ============================================================

# Import the Python libraries we need

import pandas as pd
import numpy as np

# Set a random seed so our synthetic data is reproducible
np.random.seed(42)

# Define the fictional company

company_name = "Nexus DeepTech"

starting_headcount = 250
target_headcount = 400

total_hires = 210
total_departures = 60

# Define the departments

departments = [
    "Engineering",
    "AI Research",
    "Commercial",
    "Operations",
    "G&A"
]

# Define the 12 months of our planning year

months = pd.date_range(
    start="2026-01-01",
    periods=12,
    freq="MS"
)

print("Project setup complete!")
print("Company:", company_name)
print("Starting headcount:", starting_headcount)
print("Target headcount:", target_headcount)
print("Departments:", departments)
print("Planning period:", months[0].strftime("%B %Y"),
      "to", months[-1].strftime("%B %Y"))


Project setup complete!
Company: Nexus DeepTech
Starting headcount: 250
Target headcount: 400
Departments: ['Engineering', 'AI Research', 'Commercial', 'Operations', 'G&A']
Planning period: January 2026 to December 2026


In [2]:

# ============================================================
# STEP 2: MONTHLY HIRING AND DEPARTURE PLAN
# ============================================================

# Planned monthly hires
monthly_hires = [14, 15, 16, 17, 17, 18,
                 18, 19, 19, 20, 19, 18]

# Planned monthly departures
monthly_departures = [3, 4, 4, 5, 4, 5,
                      5, 5, 6, 5, 7, 7]

# Create the monthly workforce plan
workforce_plan = pd.DataFrame({
    "Month": months,
    "Hires": monthly_hires,
    "Departures": monthly_departures
})

# Calculate net workforce movement
workforce_plan["Net_Change"] = (
    workforce_plan["Hires"] -
    workforce_plan["Departures"]
)

# Calculate monthly headcount
workforce_plan["Headcount"] = (
    starting_headcount +
    workforce_plan["Net_Change"].cumsum()
)

# Display the workforce plan
display(workforce_plan)

# Check annual totals
print("Total hires:", workforce_plan["Hires"].sum())
print("Total departures:", workforce_plan["Departures"].sum())
print("December headcount:", workforce_plan["Headcount"].iloc[-1])


,Month,Hires,Departures,Net_Change,Headcount
0,2026-01-01,14,3,11,261
1,2026-02-01,15,4,11,272
2,2026-03-01,16,4,12,284
3,2026-04-01,17,5,12,296
4,2026-05-01,17,4,13,309
5,2026-06-01,18,5,13,322
6,2026-07-01,18,5,13,335
7,2026-08-01,19,5,14,349
8,2026-09-01,19,6,13,362
9,2026-10-01,20,5,15,377


Total hires: 210
Total departures: 60
December headcount: 400


In [3]:

# ============================================================
# STEP 3: DEPARTMENTAL WORKFORCE ALLOCATION
# ============================================================

# Define the percentage allocation by department

department_weights = {
    "Engineering": 0.35,
    "AI Research": 0.25,
    "Commercial": 0.20,
    "Operations": 0.12,
    "G&A": 0.08
}

# Check that allocations add up to 100%

assert np.isclose(sum(department_weights.values()), 1.0)

# Allocate starting headcount across departments

starting_department_headcount = {
    dept: round(starting_headcount * weight)
    for dept, weight in department_weights.items()
}

# Correct any rounding difference so the total is exactly 250
starting_department_headcount["Engineering"] += (
    starting_headcount -
    sum(starting_department_headcount.values())
)

print("Starting headcount by department:")
print(starting_department_headcount)

print("\nTotal:",
      sum(starting_department_headcount.values()))


Starting headcount by department:
{'Engineering': 88, 'AI Research': 62, 'Commercial': 50, 'Operations': 30, 'G&A': 20}

Total: 250


In [4]:

# ============================================================
# STEP 3.2: MONTHLY HEADCOUNT BY DEPARTMENT
# ============================================================

department_records = []

# Start with January opening headcount by department
current_headcount = starting_department_headcount.copy()

for i, month in enumerate(months):

    total_monthly_hires = monthly_hires[i]
    total_monthly_departures = monthly_departures[i]

    # Allocate hires and departures by department
    department_hires = {
        dept: int(round(total_monthly_hires * weight))
        for dept, weight in department_weights.items()
    }

    department_departures = {
        dept: int(round(total_monthly_departures * weight))
        for dept, weight in department_weights.items()
    }

    # Adjust rounding so departmental hires equal monthly total
    department_hires["Engineering"] += (
        total_monthly_hires - sum(department_hires.values())
    )

    # Adjust rounding so departmental departures equal monthly total
    department_departures["Engineering"] += (
        total_monthly_departures -
        sum(department_departures.values())
    )

    # Calculate each department's closing headcount
    for dept in departments:

        opening = current_headcount[dept]
        hires = department_hires[dept]
        departures = department_departures[dept]
        closing = opening + hires - departures

        department_records.append({
            "Month": month,
            "Department": dept,
            "Opening_Headcount": opening,
            "Hires": hires,
            "Departures": departures,
            "Closing_Headcount": closing
        })

        current_headcount[dept] = closing

# Create the departmental workforce dataframe
department_monthly = pd.DataFrame(department_records)

# Display the first 10 records
display(department_monthly.head(10))

# Check December departmental headcount
december_headcount = (
    department_monthly[
        department_monthly["Month"] == months[-1]
    ]["Closing_Headcount"].sum()
)

print("December total headcount:", december_headcount)


,Month,Department,Opening_Headcount,Hires,Departures,Closing_Headcount
0,2026-01-01,Engineering,88,4,1,91
1,2026-01-01,AI Research,62,4,1,65
2,2026-01-01,Commercial,50,3,1,52
3,2026-01-01,Operations,30,2,0,32
4,2026-01-01,G&A,20,1,0,21
5,2026-02-01,Engineering,91,5,2,94
6,2026-02-01,AI Research,65,4,1,68
7,2026-02-01,Commercial,52,3,1,54
8,2026-02-01,Operations,32,2,0,34
9,2026-02-01,G&A,21,1,0,22


December total headcount: 400


In [5]:

# ============================================================
# STEP 4: SALARY BUDGET MODEL
# ============================================================

# Define average annual salaries by department

department_salaries = {
    "Engineering": 85000,
    "AI Research": 100000,
    "Commercial": 75000,
    "Operations": 55000,
    "G&A": 60000
}

# Add salary information to departmental workforce data

department_monthly["Average_Annual_Salary"] = (
    department_monthly["Department"].map(department_salaries)
)

# Calculate annualised salary run-rate by department

department_monthly["Salary_Run_Rate"] = (
    department_monthly["Closing_Headcount"] *
    department_monthly["Average_Annual_Salary"]
)

# Calculate monthly salary cost (annualised salary / 12)

department_monthly["Monthly_Salary_Cost"] = (
    department_monthly["Salary_Run_Rate"] / 12
)

# Display the results

display(
    department_monthly[
        ["Month", "Department", "Closing_Headcount",
         "Average_Annual_Salary", "Salary_Run_Rate",
         "Monthly_Salary_Cost"]
    ].head(10)
)

# Calculate December salary run-rate

december_salary = department_monthly[
    department_monthly["Month"] == months[-1]
]["Salary_Run_Rate"].sum()

print("December annualised salary run-rate: £",
      f"{december_salary:,.0f}")


,Month,Department,Closing_Headcount,Average_Annual_Salary,Salary_Run_Rate,Monthly_Salary_Cost
0,2026-01-01,Engineering,91,85000,7735000,644583.333333
1,2026-01-01,AI Research,65,100000,6500000,541666.666667
2,2026-01-01,Commercial,52,75000,3900000,325000.000000
3,2026-01-01,Operations,32,55000,1760000,146666.666667
4,2026-01-01,G&A,21,60000,1260000,105000.000000
5,2026-02-01,Engineering,94,85000,7990000,665833.333333
6,2026-02-01,AI Research,68,100000,6800000,566666.666667
7,2026-02-01,Commercial,54,75000,4050000,337500.000000
8,2026-02-01,Operations,34,55000,1870000,155833.333333
9,2026-02-01,G&A,22,60000,1320000,110000.000000


December annualised salary run-rate: £ 32,445,000


In [6]:

# ============================================================
# STEP 5: RECRUITER CAPACITY MODEL
# ============================================================

# Define recruiter capacity assumptions by department

recruiter_assumptions = {
    "Engineering": {"Recruiters": 3, "Capacity_Per_Recruiter": 3},
    "AI Research": {"Recruiters": 2, "Capacity_Per_Recruiter": 2},
    "Commercial": {"Recruiters": 2, "Capacity_Per_Recruiter": 3},
    "Operations": {"Recruiters": 1, "Capacity_Per_Recruiter": 3},
    "G&A": {"Recruiters": 1, "Capacity_Per_Recruiter": 2}
}

# Create a separate recruiter capacity dataset

recruiter_records = []

for _, row in department_monthly.iterrows():

    dept = row["Department"]

    recruiters = recruiter_assumptions[dept]["Recruiters"]

    capacity_per_recruiter = recruiter_assumptions[dept][
        "Capacity_Per_Recruiter"
    ]

    total_capacity = recruiters * capacity_per_recruiter

    planned_hires = row["Hires"]

    capacity_gap = planned_hires - total_capacity

    utilisation = (
        planned_hires / total_capacity
        if total_capacity > 0 else 0
    )

    recruiter_records.append({
        "Month": row["Month"],
        "Department": dept,
        "Planned_Hires": planned_hires,
        "Recruiters": recruiters,
        "Capacity_Per_Recruiter": capacity_per_recruiter,
        "Total_Capacity": total_capacity,
        "Capacity_Gap": capacity_gap,
        "Capacity_Utilisation": utilisation
    })

recruiter_capacity = pd.DataFrame(recruiter_records)

# Display the first 10 records

display(recruiter_capacity.head(10))

# Calculate total annual hiring demand and capacity

print("Annual planned hires:",
      recruiter_capacity["Planned_Hires"].sum())

print("Annual recruiter capacity:",
      recruiter_capacity["Total_Capacity"].sum())

print("Months/departments exceeding capacity:",
      (recruiter_capacity["Capacity_Gap"] > 0).sum())


,Month,Department,Planned_Hires,Recruiters,Capacity_Per_Recruiter,Total_Capacity,Capacity_Gap,Capacity_Utilisation
0,2026-01-01,Engineering,4,3,3,9,-5,0.444444
1,2026-01-01,AI Research,4,2,2,4,0,1.000000
2,2026-01-01,Commercial,3,2,3,6,-3,0.500000
3,2026-01-01,Operations,2,1,3,3,-1,0.666667
4,2026-01-01,G&A,1,1,2,2,-1,0.500000
5,2026-02-01,Engineering,5,3,3,9,-4,0.555556
6,2026-02-01,AI Research,4,2,2,4,0,1.000000
7,2026-02-01,Commercial,3,2,3,6,-3,0.500000
8,2026-02-01,Operations,2,1,3,3,-1,0.666667
9,2026-02-01,G&A,1,1,2,2,-1,0.500000


Annual planned hires: 210
Annual recruiter capacity: 288
Months/departments exceeding capacity: 4


In [7]:

# ============================================================
# STEP 5.1: IMPROVE RECRUITER CAPACITY REPORTING
# ============================================================

# Calculate capacity shortfall
recruiter_capacity["Capacity_Shortfall"] = (
    recruiter_capacity["Planned_Hires"] -
    recruiter_capacity["Total_Capacity"]
).clip(lower=0)

# Calculate spare capacity
recruiter_capacity["Spare_Capacity"] = (
    recruiter_capacity["Total_Capacity"] -
    recruiter_capacity["Planned_Hires"]
).clip(lower=0)

# Add a recruitment capacity status
recruiter_capacity["Capacity_Status"] = np.where(
    recruiter_capacity["Capacity_Shortfall"] > 0,
    "Capacity Shortfall",
    np.where(
        recruiter_capacity["Capacity_Utilisation"] >= 0.9,
        "High Utilisation",
        "Within Capacity"
    )
)

# Display any capacity issues
capacity_issues = recruiter_capacity[
    recruiter_capacity["Capacity_Status"] != "Within Capacity"
]

display(
    capacity_issues[
        ["Month", "Department", "Planned_Hires",
         "Total_Capacity", "Capacity_Utilisation",
         "Capacity_Shortfall", "Capacity_Status"]
    ]
)


,Month,Department,Planned_Hires,Total_Capacity,Capacity_Utilisation,Capacity_Shortfall,Capacity_Status
1,2026-01-01,AI Research,4,4,1.00,0,High Utilisation
6,2026-02-01,AI Research,4,4,1.00,0,High Utilisation
11,2026-03-01,AI Research,4,4,1.00,0,High Utilisation
16,2026-04-01,AI Research,4,4,1.00,0,High Utilisation
21,2026-05-01,AI Research,4,4,1.00,0,High Utilisation
26,2026-06-01,AI Research,4,4,1.00,0,High Utilisation
31,2026-07-01,AI Research,4,4,1.00,0,High Utilisation
36,2026-08-01,AI Research,5,4,1.25,1,Capacity Shortfall
39,2026-08-01,G&A,2,2,1.00,0,High Utilisation
41,2026-09-01,AI Research,5,4,1.25,1,Capacity Shortfall


In [8]:

# ============================================================
# STEP 6: SKILLS GAP ANALYSIS
# ============================================================

# Define current and required skill levels

skills_data = [
    ["Engineering", "Cloud Engineering", 3, 5],
    ["Engineering", "Software Development", 4, 5],
    ["AI Research", "Machine Learning", 3, 5],
    ["AI Research", "MLOps", 2, 5],
    ["Commercial", "Enterprise Sales", 3, 4],
    ["Commercial", "Strategic Partnerships", 2, 4],
    ["Operations", "Project Management", 3, 4],
    ["Operations", "Data Operations", 2, 4],
    ["G&A", "Financial Planning", 3, 4],
    ["G&A", "People Analytics", 2, 5]
]

# Create the skills dataframe

skills_gap = pd.DataFrame(
    skills_data,
    columns=[
        "Department",
        "Skill",
        "Current_Level",
        "Required_Level"
    ]
)

# Calculate the skills gap

skills_gap["Skills_Gap"] = (
    skills_gap["Required_Level"] -
    skills_gap["Current_Level"]
)

# Assign priority based on gap size

skills_gap["Priority"] = np.where(
    skills_gap["Skills_Gap"] >= 3,
    "High",
    np.where(
        skills_gap["Skills_Gap"] == 2,
        "Medium",
        "Low"
    )
)

# Display the skills gap analysis

display(skills_gap)

# Summarise gaps by priority

print("\nSkills gap priority summary:")
display(
    skills_gap.groupby("Priority")["Skill"]
    .count()
    .reset_index(name="Number_of_Skills")
)


,Department,Skill,Current_Level,Required_Level,Skills_Gap,Priority
0,Engineering,Cloud Engineering,3,5,2,Medium
1,Engineering,Software Development,4,5,1,Low
2,AI Research,Machine Learning,3,5,2,Medium
3,AI Research,MLOps,2,5,3,High
4,Commercial,Enterprise Sales,3,4,1,Low
5,Commercial,Strategic Partnerships,2,4,2,Medium
6,Operations,Project Management,3,4,1,Low
7,Operations,Data Operations,2,4,2,Medium
8,G&A,Financial Planning,3,4,1,Low
9,G&A,People Analytics,2,5,3,High



Skills gap priority summary:


,Priority,Number_of_Skills
0,High,2
1,Low,4
2,Medium,4


In [9]:

# ============================================================
# STEP 7: WORKFORCE PLANNING ASSUMPTIONS
# ============================================================

# Define workforce planning scenarios

planning_assumptions = pd.DataFrame([
    {
        "Scenario": "Base Case",
        "Hiring_Adjustment": 1.00,
        "Attrition_Adjustment": 1.00,
        "Salary_Inflation": 0.03,
        "Hiring_Delay_Months": 0
    },
    {
        "Scenario": "Growth Case",
        "Hiring_Adjustment": 1.15,
        "Attrition_Adjustment": 1.00,
        "Salary_Inflation": 0.05,
        "Hiring_Delay_Months": 0
    },
    {
        "Scenario": "Constrained Case",
        "Hiring_Adjustment": 0.85,
        "Attrition_Adjustment": 1.25,
        "Salary_Inflation": 0.03,
        "Hiring_Delay_Months": 2
    }
])

# Display assumptions

display(planning_assumptions)


,Scenario,Hiring_Adjustment,Attrition_Adjustment,Salary_Inflation,Hiring_Delay_Months
0,Base Case,1.00,1.00,0.03,0
1,Growth Case,1.15,1.00,0.05,0
2,Constrained Case,0.85,1.25,0.03,2


In [10]:

# ============================================================
# STEP 7.2: WORKFORCE SCENARIO MODELLING
# ============================================================

scenario_monthly_records = []
scenario_summary_records = []

# Average salary assumption for allocating scenario headcount
average_salary = sum(
    department_weights[dept] * department_salaries[dept]
    for dept in departments
)

# December departmental headcount from the original base plan
base_december = department_monthly[
    department_monthly["Month"] == months[-1]
]

base_department_counts = dict(zip(
    base_december["Department"],
    base_december["Closing_Headcount"]
))

# Helper function to allocate whole employees across departments
def allocate_headcount(total, weights):
    exact = {
        dept: total * weights[dept]
        for dept in departments
    }

    allocated = {
        dept: int(np.floor(exact[dept]))
        for dept in departments
    }

    remaining = total - sum(allocated.values())

    # Give remaining employees to departments with largest fractions
    ranking = sorted(
        departments,
        key=lambda dept: exact[dept] - allocated[dept],
        reverse=True
    )

    for dept in ranking[:remaining]:
        allocated[dept] += 1

    return allocated


# Run each scenario
for _, assumption in planning_assumptions.iterrows():

    scenario = assumption["Scenario"]

    # Adjust monthly hires and departures
    adjusted_hires = np.rint(
        np.array(monthly_hires) *
        assumption["Hiring_Adjustment"]
    ).astype(int)

    adjusted_departures = np.rint(
        np.array(monthly_departures) *
        assumption["Attrition_Adjustment"]
    ).astype(int)

    # Apply recruitment delay
    delay = int(assumption["Hiring_Delay_Months"])

    if delay > 0:
        delivered_hires = np.concatenate([
            np.zeros(delay, dtype=int),
            adjusted_hires[:-delay]
        ])
    else:
        delivered_hires = adjusted_hires.copy()

    # Calculate workforce month by month
    headcount = starting_headcount
    scenario_headcounts = []

    for i, month in enumerate(months):

        hires = int(delivered_hires[i])
        departures = int(adjusted_departures[i])

        opening = headcount
        headcount = opening + hires - departures

        scenario_headcounts.append(headcount)

        scenario_monthly_records.append({
            "Scenario": scenario,
            "Month": month,
            "Opening_Headcount": opening,
            "Hires": hires,
            "Departures": departures,
            "Closing_Headcount": headcount
        })

    # Calculate hiring totals and deferred hires
    planned_hires = int(adjusted_hires.sum())
    hires_delivered = int(delivered_hires.sum())
    deferred_hires = planned_hires - hires_delivered
    departures_total = int(adjusted_departures.sum())

    # Calculate December salary run-rate
    if scenario == "Base Case":
        department_counts = base_department_counts
    else:
        department_counts = allocate_headcount(
            headcount,
            department_weights
        )

    salary_run_rate = sum(
        department_counts[dept] * department_salaries[dept]
        for dept in departments
    )

    salary_after_inflation = (
        salary_run_rate *
        (1 + assumption["Salary_Inflation"])
    )

    scenario_summary_records.append({
        "Scenario": scenario,
        "Planned_Hires": planned_hires,
        "Hires_Delivered_In_Year": hires_delivered,
        "Deferred_Hires": deferred_hires,
        "Total_Departures": departures_total,
        "Year_End_Headcount": headcount,
        "Salary_Run_Rate_Before_Inflation": salary_run_rate,
        "Salary_Run_Rate_After_Inflation": round(
            salary_after_inflation
        )
    })


# Create the scenario datasets
scenario_monthly = pd.DataFrame(scenario_monthly_records)
scenario_summary = pd.DataFrame(scenario_summary_records)

# Display the results
print("WORKFORCE SCENARIO SUMMARY")

display(scenario_summary)

print("\nMONTHLY HEADCOUNT BY SCENARIO")

display(
    scenario_monthly[
        ["Scenario", "Month", "Closing_Headcount"]
    ].tail(12)
)


WORKFORCE SCENARIO SUMMARY


,Scenario,Planned_Hires,Hires_Delivered_In_Year,Deferred_Hires,Total_Departures,Year_End_Headcount,Salary_Run_Rate_Before_Inflation,Salary_Run_Rate_After_Inflation
0,Base Case,210,210,0,60,400,32445000,33418350
1,Growth Case,243,243,0,60,433,35120000,36876000
2,Constrained Case,177,146,31,75,321,26025000,26805750



MONTHLY HEADCOUNT BY SCENARIO


,Scenario,Month,Closing_Headcount
24,Constrained Case,2026-01-01,246
25,Constrained Case,2026-02-01,241
26,Constrained Case,2026-03-01,248
27,Constrained Case,2026-04-01,255
28,Constrained Case,2026-05-01,264
29,Constrained Case,2026-06-01,272
30,Constrained Case,2026-07-01,280
31,Constrained Case,2026-08-01,289
32,Constrained Case,2026-09-01,296
33,Constrained Case,2026-10-01,306


In [12]:
# ============================================================
# STEP 8: DATA VALIDATION
# ============================================================

validation_checks = []

# 1. Base case year-end headcount
base_year_end = scenario_summary.loc[
    scenario_summary["Scenario"] == "Base Case",
    "Year_End_Headcount"
].iloc[0]

validation_checks.append({
    "Check": "Base Case Year-End Headcount",
    "Actual": base_year_end,
    "Expected": 400,
    "Status": "PASS" if base_year_end == 400 else "CHECK"
})

# 2. Base case total hires
base_hires = scenario_summary.loc[
    scenario_summary["Scenario"] == "Base Case",
    "Hires_Delivered_In_Year"
].iloc[0]

validation_checks.append({
    "Check": "Base Case Hires",
    "Actual": base_hires,
    "Expected": 210,
    "Status": "PASS" if base_hires == 210 else "CHECK"
})

# 3. Base case total departures
base_departures = scenario_summary.loc[
    scenario_summary["Scenario"] == "Base Case",
    "Total_Departures"
].iloc[0]

validation_checks.append({
    "Check": "Base Case Departures",
    "Actual": base_departures,
    "Expected": 60,
    "Status": "PASS" if base_departures == 60 else "CHECK"
})

# 4. Check monthly headcount calculations
monthly_balance = (
    scenario_monthly["Opening_Headcount"]
    + scenario_monthly["Hires"]
    - scenario_monthly["Departures"]
)

monthly_balance_pass = (
    monthly_balance == scenario_monthly["Closing_Headcount"]
).all()

validation_checks.append({
    "Check": "Monthly Headcount Balance",
    "Actual": int(monthly_balance_pass),
    "Expected": 1,
    "Status": "PASS" if monthly_balance_pass else "CHECK"
})

# 5. Check departmental totals against company totals
department_totals = (
    department_monthly.groupby("Month")["Closing_Headcount"].sum()
    .reset_index()
)

company_totals = workforce_plan[
    ["Month", "Headcount"]
]

department_check = department_totals.merge(
    company_totals,
    on="Month"
)

department_balance_pass = (
    department_check["Closing_Headcount"]
    == department_check["Headcount"]
).all()

validation_checks.append({
    "Check": "Department Headcount Matches Company Total",
    "Actual": int(department_balance_pass),
    "Expected": 1,
    "Status": "PASS" if department_balance_pass else "CHECK"
})

# 6. Check constrained case deferred hires
constrained_deferred = scenario_summary.loc[
    scenario_summary["Scenario"] == "Constrained Case",
    "Deferred_Hires"
].iloc[0]

validation_checks.append({
    "Check": "Constrained Case Deferred Hires",
    "Actual": constrained_deferred,
    "Expected": 31,
    "Status": "PASS" if constrained_deferred == 31 else "CHECK"
})

# Display validation report
validation_results = pd.DataFrame(validation_checks)

print("DATA VALIDATION REPORT")
display(validation_results)

print("\nOVERALL RESULT:")
if (validation_results["Status"] == "PASS").all():
    print("All validation checks passed.")
else:
    print("Some checks need investigation.")

DATA VALIDATION REPORT


,Check,Actual,Expected,Status
0,Base Case Year-End Headcount,400,400,PASS
1,Base Case Hires,210,210,PASS
2,Base Case Departures,60,60,PASS
3,Monthly Headcount Balance,1,1,PASS
4,Department Headcount Matches Company Total,1,1,PASS
5,Constrained Case Deferred Hires,31,31,PASS



OVERALL RESULT:
All validation checks passed.


In [13]:
# ============================================================
# STEP 9: EXECUTIVE WORKFORCE INSIGHTS
# ============================================================

# Base case workforce figures
starting_headcount = 250
ending_headcount = 400

net_growth = ending_headcount - starting_headcount
growth_percentage = (net_growth / starting_headcount) * 100

total_hires = workforce_plan["Hires"].sum()
total_departures = workforce_plan["Departures"].sum()

# Salary cost
december_salary_cost = department_monthly[
    department_monthly["Month"] == months[-1]
]["Salary_Run_Rate"].sum()

# Recruiter capacity
total_recruiter_capacity = recruiter_capacity["Total_Capacity"].sum()
total_capacity_gap = recruiter_capacity["Capacity_Shortfall"].sum()

# Skills gaps
high_priority_gaps = skills_gap[
    skills_gap["Priority"] == "High"
]

# Display executive summary
print("NEXUS DEEPTECH — WORKFORCE EXECUTIVE SUMMARY")
print("------------------------------------------------")
print(f"Starting headcount: {starting_headcount}")
print(f"Year-end headcount: {ending_headcount}")
print(f"Net workforce growth: {net_growth}")
print(f"Workforce growth: {growth_percentage:.1f}%")
print(f"Total planned hires: {total_hires}")
print(f"Total departures: {total_departures}")
print(f"December annualised salary run-rate: £{december_salary_cost:,.0f}")
print(f"Total recruiter capacity: {total_recruiter_capacity}")
print(f"Total capacity shortfall across months: {total_capacity_gap}")

print("\nHIGH-PRIORITY SKILLS GAPS")
display(high_priority_gaps)

NEXUS DEEPTECH — WORKFORCE EXECUTIVE SUMMARY
------------------------------------------------
Starting headcount: 250
Year-end headcount: 400
Net workforce growth: 150
Workforce growth: 60.0%
Total planned hires: 210
Total departures: 60
December annualised salary run-rate: £32,445,000
Total recruiter capacity: 288
Total capacity shortfall across months: 4

HIGH-PRIORITY SKILLS GAPS


,Department,Skill,Current_Level,Required_Level,Skills_Gap,Priority
3,AI Research,MLOps,2,5,3,High
9,G&A,People Analytics,2,5,3,High


In [14]:
# ============================================================
# STEP 10: EXECUTIVE DASHBOARD DATA
# ============================================================

dashboard_summary = pd.DataFrame([
    {
        "Metric": "Starting Headcount",
        "Value": 250,
        "Category": "Workforce"
    },
    {
        "Metric": "Year-End Headcount",
        "Value": 400,
        "Category": "Workforce"
    },
    {
        "Metric": "Net Workforce Growth",
        "Value": 150,
        "Category": "Workforce"
    },
    {
        "Metric": "Workforce Growth Percentage",
        "Value": 60,
        "Category": "Workforce"
    },
    {
        "Metric": "Total Planned Hires",
        "Value": 210,
        "Category": "Recruitment"
    },
    {
        "Metric": "Total Departures",
        "Value": 60,
        "Category": "Recruitment"
    },
    {
        "Metric": "Annual Recruiter Capacity",
        "Value": 288,
        "Category": "Recruitment"
    },
    {
        "Metric": "Recruitment Capacity Shortfalls",
        "Value": 4,
        "Category": "Recruitment"
    },
    {
        "Metric": "Annualised Salary Run-Rate",
        "Value": 32445000,
        "Category": "Cost"
    },
    {
        "Metric": "High-Priority Skills Gaps",
        "Value": 2,
        "Category": "Skills"
    }
])

print("EXECUTIVE DASHBOARD SUMMARY DATA")
display(dashboard_summary)

EXECUTIVE DASHBOARD SUMMARY DATA


,Metric,Value,Category
0,Starting Headcount,250,Workforce
1,Year-End Headcount,400,Workforce
2,Net Workforce Growth,150,Workforce
3,Workforce Growth Percentage,60,Workforce
4,Total Planned Hires,210,Recruitment
5,Total Departures,60,Recruitment
6,Annual Recruiter Capacity,288,Recruitment
7,Recruitment Capacity Shortfalls,4,Recruitment
8,Annualised Salary Run-Rate,32445000,Cost
9,High-Priority Skills Gaps,2,Skills


In [15]:
# ============================================================
# STEP 11: MONTHLY WORKFORCE DASHBOARD DATA
# ============================================================

monthly_dashboard = workforce_plan[
    ["Month", "Hires", "Departures", "Net_Change", "Headcount"]
].copy()

# Add a readable month label for dashboard charts
monthly_dashboard["Month_Name"] = (
    monthly_dashboard["Month"].dt.strftime("%b %Y")
)

# Put the month label first
monthly_dashboard = monthly_dashboard[
    ["Month", "Month_Name", "Hires", "Departures",
     "Net_Change", "Headcount"]
]

print("MONTHLY WORKFORCE TREND DATA")
display(monthly_dashboard)

MONTHLY WORKFORCE TREND DATA


,Month,Month_Name,Hires,Departures,Net_Change,Headcount
0,2026-01-01,Jan 2026,14,3,11,261
1,2026-02-01,Feb 2026,15,4,11,272
2,2026-03-01,Mar 2026,16,4,12,284
3,2026-04-01,Apr 2026,17,5,12,296
4,2026-05-01,May 2026,17,4,13,309
5,2026-06-01,Jun 2026,18,5,13,322
6,2026-07-01,Jul 2026,18,5,13,335
7,2026-08-01,Aug 2026,19,5,14,349
8,2026-09-01,Sep 2026,19,6,13,362
9,2026-10-01,Oct 2026,20,5,15,377


In [16]:
# ============================================================
# STEP 12: SALARY COST DASHBOARD DATA
# ============================================================

salary_dashboard = department_monthly[
    [
        "Month",
        "Department",
        "Closing_Headcount",
        "Average_Annual_Salary",
        "Salary_Run_Rate",
        "Monthly_Salary_Cost"
    ]
].copy()

salary_dashboard["Month_Name"] = (
    salary_dashboard["Month"].dt.strftime("%b %Y")
)

salary_dashboard = salary_dashboard[
    [
        "Month",
        "Month_Name",
        "Department",
        "Closing_Headcount",
        "Average_Annual_Salary",
        "Salary_Run_Rate",
        "Monthly_Salary_Cost"
    ]
]

print("SALARY COST DASHBOARD DATA")
display(salary_dashboard.head(15))

print("\nTOTAL DECEMBER ANNUALISED SALARY RUN-RATE:")
print(
    "£{:,.0f}".format(
        salary_dashboard[
            salary_dashboard["Month"] == months[-1]
        ]["Salary_Run_Rate"].sum()
    )
)

SALARY COST DASHBOARD DATA


,Month,Month_Name,Department,Closing_Headcount,Average_Annual_Salary,Salary_Run_Rate,Monthly_Salary_Cost
0,2026-01-01,Jan 2026,Engineering,91,85000,7735000,644583.333333
1,2026-01-01,Jan 2026,AI Research,65,100000,6500000,541666.666667
2,2026-01-01,Jan 2026,Commercial,52,75000,3900000,325000.000000
3,2026-01-01,Jan 2026,Operations,32,55000,1760000,146666.666667
4,2026-01-01,Jan 2026,G&A,21,60000,1260000,105000.000000
5,2026-02-01,Feb 2026,Engineering,94,85000,7990000,665833.333333
6,2026-02-01,Feb 2026,AI Research,68,100000,6800000,566666.666667
7,2026-02-01,Feb 2026,Commercial,54,75000,4050000,337500.000000
8,2026-02-01,Feb 2026,Operations,34,55000,1870000,155833.333333
9,2026-02-01,Feb 2026,G&A,22,60000,1320000,110000.000000



TOTAL DECEMBER ANNUALISED SALARY RUN-RATE:
£32,445,000


In [17]:
# ============================================================
# STEP 13: RECRUITMENT CAPACITY DASHBOARD DATA
# ============================================================

recruitment_dashboard = recruiter_capacity.copy()

recruitment_dashboard["Month_Name"] = (
    recruitment_dashboard["Month"].dt.strftime("%b %Y")
)

recruitment_dashboard = recruitment_dashboard[
    [
        "Month",
        "Month_Name",
        "Department",
        "Planned_Hires",
        "Recruiters",
        "Total_Capacity",
        "Capacity_Shortfall",
        "Spare_Capacity",
        "Capacity_Utilisation",
        "Capacity_Status"
    ]
]

print("RECRUITMENT CAPACITY DASHBOARD DATA")
display(recruitment_dashboard.head(15))

print("\nCAPACITY STATUS SUMMARY")
display(
    recruitment_dashboard["Capacity_Status"]
    .value_counts()
    .reset_index()
)

RECRUITMENT CAPACITY DASHBOARD DATA


,Month,Month_Name,Department,Planned_Hires,Recruiters,Total_Capacity,Capacity_Shortfall,Spare_Capacity,Capacity_Utilisation,Capacity_Status
0,2026-01-01,Jan 2026,Engineering,4,3,9,0,5,0.444444,Within Capacity
1,2026-01-01,Jan 2026,AI Research,4,2,4,0,0,1.000000,High Utilisation
2,2026-01-01,Jan 2026,Commercial,3,2,6,0,3,0.500000,Within Capacity
3,2026-01-01,Jan 2026,Operations,2,1,3,0,1,0.666667,Within Capacity
4,2026-01-01,Jan 2026,G&A,1,1,2,0,1,0.500000,Within Capacity
5,2026-02-01,Feb 2026,Engineering,5,3,9,0,4,0.555556,Within Capacity
6,2026-02-01,Feb 2026,AI Research,4,2,4,0,0,1.000000,High Utilisation
7,2026-02-01,Feb 2026,Commercial,3,2,6,0,3,0.500000,Within Capacity
8,2026-02-01,Feb 2026,Operations,2,1,3,0,1,0.666667,Within Capacity
9,2026-02-01,Feb 2026,G&A,1,1,2,0,1,0.500000,Within Capacity



CAPACITY STATUS SUMMARY


,Capacity_Status,count
0,Within Capacity,44
1,High Utilisation,12
2,Capacity Shortfall,4


In [18]:
# ============================================================
# STEP 14: SKILLS GAP DASHBOARD DATA
# ============================================================

skills_dashboard = skills_gap.copy()

skills_dashboard["Gap_Label"] = np.where(
    skills_dashboard["Skills_Gap"] > 0,
    "Skills Gap",
    "No Gap"
)

print("SKILLS GAP DASHBOARD DATA")
display(skills_dashboard)

print("\nSKILLS GAP SUMMARY BY PRIORITY")
display(
    skills_dashboard.groupby("Priority")
    .size()
    .reset_index(name="Number_of_Skills")
    .sort_values("Priority")
)

print("\nSKILLS GAP SUMMARY BY DEPARTMENT")
display(
    skills_dashboard.groupby("Department")["Skills_Gap"]
    .sum()
    .reset_index(name="Total_Skills_Gap")
    .sort_values("Total_Skills_Gap", ascending=False)
)

SKILLS GAP DASHBOARD DATA


,Department,Skill,Current_Level,Required_Level,Skills_Gap,Priority,Gap_Label
0,Engineering,Cloud Engineering,3,5,2,Medium,Skills Gap
1,Engineering,Software Development,4,5,1,Low,Skills Gap
2,AI Research,Machine Learning,3,5,2,Medium,Skills Gap
3,AI Research,MLOps,2,5,3,High,Skills Gap
4,Commercial,Enterprise Sales,3,4,1,Low,Skills Gap
5,Commercial,Strategic Partnerships,2,4,2,Medium,Skills Gap
6,Operations,Project Management,3,4,1,Low,Skills Gap
7,Operations,Data Operations,2,4,2,Medium,Skills Gap
8,G&A,Financial Planning,3,4,1,Low,Skills Gap
9,G&A,People Analytics,2,5,3,High,Skills Gap



SKILLS GAP SUMMARY BY PRIORITY


,Priority,Number_of_Skills
0,High,2
1,Low,4
2,Medium,4



SKILLS GAP SUMMARY BY DEPARTMENT


,Department,Total_Skills_Gap
0,AI Research,5
3,G&A,4
1,Commercial,3
2,Engineering,3
4,Operations,3


In [19]:
# ============================================================
# STEP 15: EXPORT PORTFOLIO DATASETS
# ============================================================

dashboard_summary.to_csv(
    "executive_dashboard_summary.csv", index=False
)

monthly_dashboard.to_csv(
    "monthly_workforce_trends.csv", index=False
)

salary_dashboard.to_csv(
    "salary_cost_dashboard.csv", index=False
)

recruitment_dashboard.to_csv(
    "recruitment_capacity_dashboard.csv", index=False
)

skills_dashboard.to_csv(
    "skills_gap_dashboard.csv", index=False
)

scenario_summary.to_csv(
    "workforce_scenario_summary.csv", index=False
)

scenario_monthly.to_csv(
    "workforce_scenario_monthly.csv", index=False
)

validation_results.to_csv(
    "data_validation_report.csv", index=False
)

print("All 8 portfolio datasets exported successfully.")

All 8 portfolio datasets exported successfully.
